In [ ]:
# ============================================================================
# HAB Toxin Production - ML Classification Analysis
# Models : Logistic Regression | Decision Tree | XGBoost | Random Forest | ANN
# Protocol: 5-Fold × 10-Repeat Stratified CV  +  Hyperparameter Tuning
#
# Hyperparameter search — penalty / search strategy per model:
#   ┌──────────────────────┬──────────────────────────────────────────────────┐
#   │ Logistic Regression  │ NO penalty  (plain accuracy)                     │
#   │                      │  • RandomizedSearchCV, accuracy scorer           │
#   │                      │  • n_iter = 500  (expanded l1/l2 grid)           │
#   ├──────────────────────┼──────────────────────────────────────────────────┤
#   │ Decision Tree        │ NO penalty  (plain accuracy)                     │
#   │                      │  • RandomizedSearchCV, accuracy scorer           │
#   │                      │  • n_iter = 2500                                 │
#   │                      │  • No gap penalty applied during search          │
#   ├──────────────────────┼──────────────────────────────────────────────────┤
#   │ XGBoost              │ HARD penalty  (accuracy-based)                   │
#   │ Random Forest        │  • Returns -999 if gap > GAP_TOLERANCE (10%)     │
#   │                      │  • Fallback: re-run at GAP_FALLBACK_TOLERANCE    │
#   │                      │    (12%) if ALL candidates fail                  │
#   ├──────────────────────┼──────────────────────────────────────────────────┤
#   │ ANN                  │ NO penalty  (plain val_accuracy)                 │
#   │                      │  • Manual grid loop over hidden_layers, neurons, │
#   │                      │    activation, learning_rate, dropout_rate       │
#   │                      │  • Selection: best mean val_accuracy across      │
#   │                      │    3 inner stratified CV folds                   │
#   └──────────────────────┴──────────────────────────────────────────────────┘
#
#   All RandomizedSearchCV scorers:
#     • Primary metric : accuracy on TEST folds
#     • Gap measured   : 3-fold internal stratified CV per candidate
#                        (RF/XGB hard penalty only)
#     • n_iter         : LR=500, DT=2500, RF=2500, XGB=2500
#
#   ANN manual grid:
#     • hidden_layers  : [2, 3, 4]
#     • activations    : ['relu', 'elu', 'sigmoid', 'tanh']
#     • neurons        : [16, 32, 64, 128]
#     • learning_rates : [0.0001, 0.001, 0.005, 0.0081, 0.01, 0.05, 0.1]
#     • dropout_rates  : [0.1, 0.15, 0.2, 0.25, 0.3]
#     • Selection: best mean val_accuracy (3-fold inner CV)
#
#   Best params are fixed and reused across all 50 outer CV folds.
#   Best params + tuning scores saved to tables/hyperparameter_tuning.csv.
#
#   Model order (consistent with ML_hab_modeling_analysis2_RF_XGB_hard_penalty1.py):
#     Logistic Regression → Decision Tree → XGBoost → Random Forest → ANN
#
# Generates:
#   (1) Model comparison bar plots  — train & test mean ± 1 SD across 50 folds
#                                     individual per-metric + combined 2×3 figure
#   (2) Confusion matrices          — aggregated per-fold test predictions
#                                     individual per-model + combined figure
#   (3) hyperparameter_tuning.csv   — best params + diagnostic scores per model
#   (4) metrics_summary.csv         — train/test mean ± SD for all metrics
#
# CI approach: mean ± 1 SD of per-fold scores (50 folds)
# ============================================================================

import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
# Run from the notebooks/ directory OR set kernel working directory to HAB_Phase1/
import sys, pathlib


import gc
import random
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.ioff()
import matplotlib.patches as mpatches

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import (
    RepeatedStratifiedKFold, RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, make_scorer
)
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform, loguniform
import tensorflow as tf
import tensorflow.keras as keras

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION  — edit paths / columns / hyperparameters here
# ============================================================================

PROJECT_ROOT = pathlib.Path().resolve()
while PROJECT_ROOT.name != "HAB_Phase1":
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find HAB_Phase1 project root")

sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, OUTPUT_DIRS

# DATA_PATH    = "Data/UCDavis_Data_for_Analysis2_220days_ToxinProdMicrocystis.csv"
# OUTPUT_DIR   = "C:/Temp/HS_CV_5ML"
OUTPUT_DIR = str(OUTPUT_DIRS["03"])  # change for each notebook
DATA_PATH  = str(DATA_PATH)

TARGET_COL   = "ToxinProdc_Microcystis_CC"
SITE_COL     = "Site_Name"
DATE_COL     = "Date"
THRESHOLD    = 4000
RANDOM_STATE = 42

FEATURES = ["Q", "Q_1m", "SpecCond.", "OrgC", "OrgN", "PO4",
            "WaterTemp", "DO", "pH"]

# ============================================================================
# MIXED-PENALTY / SEARCH STRATEGY CONSTANTS
# ============================================================================
#
# Shared gap thresholds (RF/XGB hard penalty)
GAP_TOLERANCE          = 0.10    # primary gap threshold
GAP_FALLBACK_TOLERANCE = 0.12    # fallback threshold (RF/XGB hard only)
HARD_PENALTY_SCORE     = -999.0  # hard-disqualification score (RF, XGB)

# Inner CV folds used to estimate gap per candidate in scorer
SCORER_CV_FOLDS = 3

# Outer folds used inside RandomizedSearchCV
TUNE_CV_FOLDS = 5

# n_iter per model
TUNE_N_ITER_LR  = 5 #500
TUNE_N_ITER_DT  = 5 #2500
TUNE_N_ITER_RF  = 5 #2500
TUNE_N_ITER_XGB = 5 #2500

# Module-level mutable state — updated before each search run so the scorer
# callables can read current values without needing extra arguments.
_ACTIVE_GAP_TOLERANCE = GAP_TOLERANCE

# ============================================================================
# REPEATED CV SETTINGS
# ============================================================================
N_SPLITS  = 5
N_REPEATS = 10
# Total folds = N_SPLITS * N_REPEATS = 50

# Decision threshold for all classifiers
DECISION_THR = 0.5

# ============================================================================
# MODEL ORDER  (consistent with ML_hab_modeling_analysis2_RF_XGB_hard_penalty1)
# ============================================================================
ALL_MODELS = ["Logistic Regression", "Decision Tree", "XGBoost",
              "Random Forest", "ANN"]

# ============================================================================
# COLORS
# ============================================================================
COLORS = {
    "Logistic Regression": "#9467BD",
    "Decision Tree":       "#8C564B",
    "XGBoost":             "#B85042",
    "Random Forest":       "#1C7293",
    "ANN":                 "#E8A838",
}
TRAIN_COLOR = "#607D8B"
TEST_COLOR  = "#2E4057"


# ============================================================================
# OUTPUT DIRECTORIES
# ============================================================================
def setup_output_dirs():
    subdirs = [
        "figures/model_comparison",
        "figures/confusion_matrix",
        "tables",
        "logs",
    ]
    for sub in subdirs:
        path = os.path.join(OUTPUT_DIR, sub)
        os.makedirs(path, exist_ok=True)
    print("\nOutput directories created successfully")
    print("Output directory:", os.path.abspath(OUTPUT_DIR))


def savefig(fig, name):
    """Save figure as PNG and EPS."""
    try:
        name   = name.replace("\\", "/")
        folder = os.path.abspath(
            os.path.join(OUTPUT_DIR, "figures", os.path.dirname(name))
        )
        os.makedirs(folder, exist_ok=True)
        filename = os.path.basename(name)
        png_path = os.path.join(folder, filename + ".png")
        eps_path = os.path.join(folder, filename + ".eps")
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        try:
            fig.savefig(eps_path, format="eps", bbox_inches="tight")
        except Exception as e:
            print(f"  EPS save failed for {filename}: {e}")
        print(f"  Saved → {png_path}")
    except Exception as e:
        print(f"\n  ERROR SAVING FIGURE '{name}': {e}")


# ============================================================================
# SEED
# ============================================================================
def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ============================================================================
# INNER CV ACCURACY HELPER  (shared by all gap-aware scorers)
# ============================================================================
def _inner_cv_accuracy(estimator, X, y):
    """
    Fit a clone of estimator on SCORER_CV_FOLDS=3 internal stratified folds
    and return (train_acc_list, test_acc_list, exception_occurred).
    """
    skf = StratifiedKFold(n_splits=SCORER_CV_FOLDS, shuffle=True,
                          random_state=RANDOM_STATE)
    train_acc_list, test_acc_list = [], []
    try:
        for tr_i, te_i in skf.split(X, y):
            est_clone = clone(estimator)
            est_clone.fit(X[tr_i], y[tr_i])
            train_acc_list.append(accuracy_score(y[tr_i],
                                                 est_clone.predict(X[tr_i])))
            test_acc_list.append(accuracy_score(y[te_i],
                                                est_clone.predict(X[te_i])))
        return train_acc_list, test_acc_list, False
    except Exception:
        return [], [], True


# ============================================================================
# SCORER: LR — plain accuracy (no penalty)
# ============================================================================
def _plain_accuracy(estimator, X, y):
    """
    Plain accuracy scorer for Logistic Regression.
    No gap penalty applied — uses 3-fold inner CV test accuracy directly.
    On error: returns 0.0.
    """
    _, test_acc_list, error = _inner_cv_accuracy(estimator, X, y)
    if error or len(test_acc_list) == 0:
        return 0.0
    return np.mean(test_acc_list)


# ============================================================================
# SCORER: RF / XGB — HARD penalty
# ============================================================================
def _hard_penalty_accuracy(estimator, X, y):
    """
    Hard-penalty accuracy scorer for Random Forest and XGBoost.
    Uses SCORER_CV_FOLDS=3 internal splits to compute gap.
    Decision rule:
        gap <= _ACTIVE_GAP_TOLERANCE  →  return mean(test_accuracy)
        gap >  _ACTIVE_GAP_TOLERANCE  →  return HARD_PENALTY_SCORE (-999)
    On error: returns HARD_PENALTY_SCORE.
    """
    train_acc_list, test_acc_list, error = _inner_cv_accuracy(estimator, X, y)
    if error or len(train_acc_list) == 0:
        return HARD_PENALTY_SCORE
    gap = np.mean(train_acc_list) - np.mean(test_acc_list)
    if gap > _ACTIVE_GAP_TOLERANCE:
        return HARD_PENALTY_SCORE
    return np.mean(test_acc_list)


# Assign scorer aliases
LR_SCORING    = _plain_accuracy
DT_SCORING    = _plain_accuracy   # no gap penalty for DT
RFXGB_SCORING = _hard_penalty_accuracy


# ============================================================================
# HYPERPARAMETER GRIDS
# ============================================================================

# ── Logistic Regression ───────────────────────────────────────────────────────
# Expanded grid: covers l1, l2, elasticnet; multiple solvers, C values,
# class_weight, tol, max_iter variations.
# Finalized params from analysis file: solver="saga", penalty="l1", C=1,
#   class_weight="balanced", max_iter=1000, tol=0.0001
LR_PARAM_DIST = [
    # l2 — compatible solvers: lbfgs, saga, newton-cg, sag
    {
        "penalty":       ["l2"],
        "C":             [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100],
        "solver":        ["lbfgs", "saga", "sag", "newton-cg"],
        "max_iter":      [500, 1000, 2000],
        "tol":           [1e-5, 1e-4, 1e-3],
        "class_weight":  ["balanced", None],
        "fit_intercept": [True],
        "warm_start":    [False],
    },
    # l1 — compatible solvers: liblinear, saga
    {
        "penalty":       ["l1"],
        "C":             [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100],
        "solver":        ["liblinear", "saga"],
        "max_iter":      [500, 1000, 2000],
        "tol":           [1e-5, 1e-4, 1e-3],
        "class_weight":  ["balanced", None],
        "fit_intercept": [True],
        "warm_start":    [False],
    },
    # elasticnet — compatible solver: saga only; requires l1_ratio
    {
        "penalty":       ["elasticnet"],
        "C":             [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10],
        "solver":        ["saga"],
        "l1_ratio":      [0.1, 0.25, 0.5, 0.75, 0.9],
        "max_iter":      [500, 1000, 2000],
        "tol":           [1e-5, 1e-4, 1e-3],
        "class_weight":  ["balanced", None],
        "fit_intercept": [True],
        "warm_start":    [False],
    },
    # None penalty (intercept-only baseline)
    {
        "penalty":       [None],
        "C":             [1.0],             # C ignored when penalty=None
        "solver":        ["lbfgs", "saga", "newton-cg"],
        "max_iter":      [500, 1000],
        "tol":           [1e-4],
        "class_weight":  ["balanced", None],
        "fit_intercept": [True, False],
        "warm_start":    [False],
    },
]

# ── Decision Tree ─────────────────────────────────────────────────────────────
# Finalized params from analysis file:
#   splitter="random", min_samples_split=5, min_samples_leaf=2,
#   max_features=0.5, max_depth=5, criterion="entropy",
#   class_weight="balanced", ccp_alpha=0.01, min_impurity_decrease=0.0
DT_PARAM_DIST = {
    "criterion":                ["gini", "entropy"],
    "max_depth":                [3, 5, 7, 10, 15],       # finalized: 5
    "min_samples_split":        [2, 5, 10, 20],           # finalized: 5
    "min_samples_leaf":         [1, 2, 4, 6, 10],         # finalized: 2
    "max_features":             ["sqrt", "log2", 0.3, 0.5, 0.7],  # finalized: 0.5
    "splitter":                 ["best", "random"],       # finalized: "random"
    "class_weight":             ["balanced", None],       # finalized: "balanced"
    "ccp_alpha":                loguniform(1e-4, 0.1),    # finalized: 0.01
    "min_impurity_decrease":    uniform(0.0, 0.05),       # finalized: 0.0
    "min_weight_fraction_leaf": uniform(0.0, 0.03),
    "max_leaf_nodes":           [None, 10, 20, 30, 50],
}

# ── Random Forest ─────────────────────────────────────────────────────────────
# Finalized params from analysis file:
#   n_estimators=300, max_depth=5, min_samples_split=5, min_samples_leaf=8,
#   max_features=0.3, bootstrap=True, class_weight="balanced",
#   criterion="gini", ccp_alpha=0.0001, min_impurity_decrease=0.01627,
#   min_weight_fraction_leaf=0.01946, max_leaf_nodes=57

_RF_SHARED = dict(
    n_estimators             = [100, 200, 300, 500],      # finalized: 300
    max_depth                = [3, 5, 7, 10, 15],         # finalized: 5
    min_samples_split        = [2, 5, 10],                # finalized: 5
    min_samples_leaf         = [2, 4, 6, 8, 12],          # finalized: 8
    max_features             = ["sqrt", "log2", 0.3, 0.5],# finalized: 0.3
    criterion                = ["gini", "entropy"],       # finalized: "gini"
    min_impurity_decrease    = uniform(0.0, 0.03),        # finalized: 0.01627
    ccp_alpha                = loguniform(1e-4, 0.05),    # finalized: 0.0001
    min_weight_fraction_leaf = uniform(0.0, 0.03),        # finalized: 0.01946
    max_leaf_nodes           = randint(20, 100),          # finalized: 57
)

_RF_BOOT_TRUE = {
    **_RF_SHARED,
    "bootstrap":    [True],                               # finalized: True
    "max_samples":  [0.6, 0.7, 0.8, 0.9],
    "oob_score":    [False],
    "class_weight": ["balanced", "balanced_subsample", None],  # finalized: "balanced"
}

_RF_BOOT_FALSE = {
    **_RF_SHARED,
    "bootstrap":    [False],
    "max_samples":  [None],
    "oob_score":    [False],
    "class_weight": ["balanced", None],
}

RF_PARAM_DIST = [_RF_BOOT_TRUE, _RF_BOOT_FALSE]

# ── XGBoost ───────────────────────────────────────────────────────────────────
# Finalized params from analysis file:
#   subsample=0.8, scale_pos_weight=1.0, n_estimators=200,
#   min_child_weight=3, max_depth=2, learning_rate=0.05,
#   reg_lambda=1.0, gamma=0.3, colsample_bytree=0.5,
#   alpha=0.1, colsample_bylevel=0.8278, colsample_bynode=0.7501,
#   max_delta_step=0
XGB_PARAM_DIST = {
    "learning_rate":     [0.005, 0.01, 0.05, 0.1],       # finalized: 0.05
    "n_estimators":      [100, 150, 200, 300],            # finalized: 200
    "max_depth":         [2, 3, 4, 5],                    # finalized: 2
    "min_child_weight":  [1, 3, 5, 7, 10],                # finalized: 3
    "gamma":             [0.0, 0.1, 0.3, 0.5, 1.0, 2.0], # finalized: 0.3
    "subsample":         [0.5, 0.6, 0.7, 0.8, 0.9],      # finalized: 0.8
    "colsample_bytree":  [0.4, 0.5, 0.6, 0.7, 0.8],      # finalized: 0.5
    "reg_lambda":        [0.5, 1.0, 2.0, 3.0, 5.0],      # finalized: 1.0
    "alpha":             [0.0, 0.1, 0.5, 1.0, 2.0],       # finalized: 0.1 (L1)
    "scale_pos_weight":  [0.5, 1.0, 2.0, 3.0, 5.0, 7.0], # finalized: 1.0
    "colsample_bylevel": uniform(0.6, 0.3),               # finalized: 0.8278
    "colsample_bynode":  uniform(0.6, 0.3),               # finalized: 0.7501
    "max_delta_step":    randint(0, 5),                   # finalized: 0
    "reg_alpha_cont":    loguniform(1e-4, 2.0),           # continuous L1 sweep alias
    "reg_lambda_cont":   loguniform(0.5, 5.0),            # continuous L2 sweep alias
}

# ── ANN manual grid ───────────────────────────────────────────────────────────
# Finalized params from analysis file:
#   hidden_layers=3, activation="elu", neurons=128, learning_rate=0.0081,
#   dropout_rate=0.25, epochs=1000, batch_size=32, patience=30
ANN_GRID = {
    "hidden_layers":  [2, 3, 4],                              # finalized: 3
    "activations":    ["relu", "elu", "sigmoid", "tanh"],     # finalized: "elu"
    "neurons":        [16, 32, 64, 128],                      # finalized: 128
    "learning_rates": [0.0001, 0.001, 0.005, 0.0081, 0.01, 0.05, 0.1],  # finalized: 0.0081
    "dropout_rates":  [0.1, 0.15, 0.2, 0.25, 0.3],           # finalized: 0.25
}
ANN_EPOCHS     = 5 #1000
ANN_BATCH_SIZE = 32
ANN_VAL_SPLIT  = 0.2
ANN_PATIENCE   = 30
ANN_INNER_FOLDS = 3   # inner folds for ANN grid search (same as SCORER_CV_FOLDS)


# ============================================================================
# MODEL BUILDERS  (always return fresh, unfitted instances)
# ============================================================================
def build_lr(best_params: dict = None):
    """
    Build a LogisticRegression with conservative defaults.
    Finalized: solver="saga", penalty="l1", C=1, class_weight="balanced".
    If best_params provided, those values override.
    Special handling: l1_ratio required for elasticnet; ignored otherwise.
    """
    defaults = dict(
        penalty      = "l1",
        C            = 1.0,
        solver       = "saga",
        max_iter     = 1000,
        tol          = 1e-4,
        class_weight = "balanced",
        fit_intercept= True,
        warm_start   = False,
        random_state = RANDOM_STATE,
    )
    if best_params:
        bp = dict(best_params)
        defaults.update(bp)
    # elasticnet requires l1_ratio; other penalties must not have it set
    if defaults.get("penalty") != "elasticnet":
        defaults.pop("l1_ratio", None)
    elif "l1_ratio" not in defaults:
        defaults["l1_ratio"] = 0.5   # safe fallback if somehow missing
    return LogisticRegression(**defaults)


def build_dt(best_params: dict = None):
    """
    Build a DecisionTreeClassifier with conservative defaults.
    Finalized: splitter="random", min_samples_split=5, min_samples_leaf=2,
    max_features=0.5, max_depth=5, criterion="entropy",
    class_weight="balanced", ccp_alpha=0.01.
    """
    defaults = dict(
        criterion                = "entropy",
        max_depth                = 5,
        min_samples_split        = 5,
        min_samples_leaf         = 2,
        max_features             = 0.5,
        splitter                 = "random",
        class_weight             = "balanced",
        ccp_alpha                = 0.01,
        min_impurity_decrease    = 0.0,
        min_weight_fraction_leaf = 0.0,
        random_state             = RANDOM_STATE,
    )
    if best_params:
        defaults.update(best_params)
    return DecisionTreeClassifier(**defaults)


def build_rf(best_params: dict = None):
    """
    Build a RandomForestClassifier with conservative defaults.
    Finalized: n_estimators=300, max_depth=5, min_samples_split=5,
    min_samples_leaf=8, max_features=0.3, bootstrap=True,
    class_weight="balanced", criterion="gini", ccp_alpha=0.0001,
    min_impurity_decrease=0.01627, min_weight_fraction_leaf=0.01946,
    max_leaf_nodes=57.
    """
    defaults = dict(
        n_estimators             = 300,
        max_depth                = 5,
        min_samples_split        = 5,
        min_samples_leaf         = 8,
        max_features             = 0.3,
        bootstrap                = True,
        class_weight             = "balanced",
        criterion                = "gini",
        max_samples              = 0.8,
        oob_score                = False,
        ccp_alpha                = 0.0001,
        min_impurity_decrease    = 0.01627,
        min_weight_fraction_leaf = 0.01946,
        max_leaf_nodes           = 57,
        random_state             = RANDOM_STATE,
    )
    if best_params:
        defaults.update(best_params)
    # Enforce bootstrap-dependent constraints
    if not defaults.get("bootstrap", True):
        defaults["max_samples"] = None
        defaults["oob_score"]   = False
        if defaults.get("class_weight") == "balanced_subsample":
            defaults["class_weight"] = "balanced"
    return RandomForestClassifier(**defaults)


def build_xgb(best_params: dict = None):
    """
    Build an XGBClassifier with conservative defaults.
    Finalized: subsample=0.8, n_estimators=200, min_child_weight=3,
    max_depth=2, learning_rate=0.05, reg_lambda=1.0, gamma=0.3,
    colsample_bytree=0.5, alpha=0.1, colsample_bylevel=0.8278,
    colsample_bynode=0.7501, max_delta_step=0.
    Special handling: reg_alpha_cont → alpha; reg_lambda_cont → reg_lambda.
    """
    defaults = dict(
        n_estimators     = 200,
        max_depth        = 2,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.5,
        colsample_bylevel= 0.8278,
        colsample_bynode = 0.7501,
        min_child_weight = 3,
        gamma            = 0.3,
        reg_lambda       = 1.0,
        alpha            = 0.1,
        scale_pos_weight = 1.0,
        max_delta_step   = 0,
        booster          = "gbtree",
        tree_method      = "hist",
        eval_metric      = "logloss",
        objective        = "binary:logistic",
        random_state     = RANDOM_STATE,
        verbosity        = 0,
        n_jobs           = 1,
    )
    if best_params:
        bp = dict(best_params)
        # Collapse continuous-sweep aliases
        if "reg_alpha_cont" in bp:
            bp.setdefault("alpha",      bp.pop("reg_alpha_cont"))
        else:
            bp.pop("reg_alpha_cont", None)
        if "reg_lambda_cont" in bp:
            bp.setdefault("reg_lambda", bp.pop("reg_lambda_cont"))
        else:
            bp.pop("reg_lambda_cont", None)
        defaults.update(bp)
    return XGBClassifier(**defaults)


def build_ann(input_dim: int,
              hidden_layers: int   = 3,
              activation: str      = "elu",
              neurons: int         = 128,
              learning_rate: float = 0.0081,
              dropout_rate: float  = 0.25) -> keras.Model:
    """
    Build a Keras binary classifier.
    Finalized: hidden_layers=3, activation="elu", neurons=128,
    learning_rate=0.0081, dropout_rate=0.25.
    """
    model = keras.Sequential(name="ANN_HAB")
    model.add(keras.layers.Input(shape=(input_dim,)))
    model.add(keras.layers.Dense(neurons, activation=activation))
    model.add(keras.layers.Dropout(dropout_rate))
    for _ in range(hidden_layers - 1):
        model.add(keras.layers.Dense(neurons, activation=activation))
        model.add(keras.layers.Dropout(dropout_rate))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ============================================================================
# ANN WRAPPER  (sklearn-compatible interface)
# ============================================================================
class ANNWrapper:
    """Thin sklearn-compatible wrapper around a Keras binary classifier."""

    def __init__(self, input_dim: int,
                 hidden_layers: int   = 3,
                 activation: str      = "elu",
                 neurons: int         = 128,
                 learning_rate: float = 0.0081,
                 dropout_rate: float  = 0.25):
        self.input_dim    = input_dim
        self.hidden_layers= hidden_layers
        self.activation   = activation
        self.neurons      = neurons
        self.learning_rate= learning_rate
        self.dropout_rate = dropout_rate
        self.model        = None

    def fit(self, X, y):
        set_global_seed(RANDOM_STATE)
        self.model = build_ann(
            self.input_dim,
            hidden_layers = self.hidden_layers,
            activation    = self.activation,
            neurons       = self.neurons,
            learning_rate = self.learning_rate,
            dropout_rate  = self.dropout_rate,
        )
        cb = keras.callbacks.EarlyStopping(
            patience=ANN_PATIENCE, restore_best_weights=True)
        self.model.fit(
            X, y,
            epochs          = ANN_EPOCHS,
            batch_size      = ANN_BATCH_SIZE,
            validation_split= ANN_VAL_SPLIT,
            callbacks       = [cb],
            verbose         = 0,
        )
        self.classes_        = np.array([0, 1])
        self.n_features_in_  = X.shape[1]
        return self

    def predict(self, X):
        probs = self.model.predict(X, verbose=0).ravel()
        return (probs >= DECISION_THR).astype(int)

    def predict_proba(self, X):
        p = np.clip(self.model.predict(X, verbose=0).ravel(), 1e-7, 1 - 1e-7)
        return np.column_stack([1 - p, p])

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


# ============================================================================
# METRIC HELPER
# ============================================================================
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":          accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision":         precision_score(y_true, y_pred, zero_division=0),
        "recall":            recall_score(y_true, y_pred, zero_division=0),
        "f1":                f1_score(y_true, y_pred, zero_division=0),
    }


# ============================================================================
# DATA LOADING
# ============================================================================
def load_data():
    print("Loading data ...")
    df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
    df = df.dropna(subset=FEATURES + [TARGET_COL])
    df["label"] = (df[TARGET_COL] >= THRESHOLD).astype(int)
    X = df[FEATURES]
    y = df["label"].values
    print(f"  Samples: {len(df)}  |  Positive (Risky): {y.sum()}  "
          f"|  Negative: {(1-y).sum()}")
    return X, y


# ============================================================================
# SEARCH HELPERS
# ============================================================================
def _run_single_search(estimator, param_dist, n_iter, X_vals, y,
                       scoring_fn) -> tuple:
    """
    Run one RandomizedSearchCV pass and return
    (rscv, best_score, all_hard_failed).
    all_hard_failed is True only for hard-penalty models where every candidate
    returned HARD_PENALTY_SCORE.
    """
    rscv = RandomizedSearchCV(
        estimator          = estimator,
        param_distributions= param_dist,
        n_iter             = n_iter,
        scoring            = scoring_fn,
        cv                 = TUNE_CV_FOLDS,
        refit              = True,
        n_jobs             = -1,
        random_state       = RANDOM_STATE,
        verbose            = 0,
        return_train_score = False,
        error_score        = HARD_PENALTY_SCORE,
    )
    rscv.fit(X_vals, y)
    best_score       = rscv.cv_results_["mean_test_score"][rscv.best_index_]
    all_hard_failed  = bool(best_score <= HARD_PENALTY_SCORE + 1.0)
    return rscv, best_score, all_hard_failed


def _diagnostic_cv(best_est, X_vals, y) -> tuple:
    """
    Manual TUNE_CV_FOLDS-fold CV on the best estimator.
    Returns (mean_train_acc, mean_test_acc, actual_gap).
    """
    diag_skf = StratifiedKFold(n_splits=TUNE_CV_FOLDS, shuffle=True,
                                random_state=RANDOM_STATE)
    tr_list, te_list = [], []
    for tr_i, te_i in diag_skf.split(X_vals, y):
        est_tmp = clone(best_est)
        est_tmp.fit(X_vals[tr_i], y[tr_i])
        tr_list.append(accuracy_score(y[tr_i], est_tmp.predict(X_vals[tr_i])))
        te_list.append(accuracy_score(y[te_i], est_tmp.predict(X_vals[te_i])))
    mean_train = np.mean(tr_list)
    mean_test  = np.mean(te_list)
    return mean_train, mean_test, mean_train - mean_test


# ============================================================================
# ANN MANUAL GRID SEARCH
# Iterates over all combinations in ANN_GRID.
# Selection criterion: best mean val_accuracy over ANN_INNER_FOLDS=3
# stratified inner CV folds.
# ============================================================================
def run_ann_grid_search(X_scaled: np.ndarray, y: np.ndarray) -> dict:
    """
    Manual grid search for ANN hyperparameters using inner stratified CV.

    For each combination of (hidden_layers, activation, neurons,
    learning_rate, dropout_rate):
      - Fit the ANN on ANN_INNER_FOLDS stratified inner folds.
      - Record mean val_accuracy across folds.
    Returns the best hyperparameter dict.
    """
    keys   = ["hidden_layers", "activations", "neurons",
              "learning_rates", "dropout_rates"]
    combos = list(itertools.product(
        ANN_GRID["hidden_layers"],
        ANN_GRID["activations"],
        ANN_GRID["neurons"],
        ANN_GRID["learning_rates"],
        ANN_GRID["dropout_rates"],
    ))
    n_combos   = len(combos)
    input_dim  = X_scaled.shape[1]
    inner_skf  = StratifiedKFold(n_splits=ANN_INNER_FOLDS, shuffle=True,
                                 random_state=RANDOM_STATE)

    print(f"\n  [ANN grid search] {n_combos} combinations  "
          f"× {ANN_INNER_FOLDS} inner folds  "
          f"= {n_combos * ANN_INNER_FOLDS} ANN fits ...")
    print(f"  Grid: hidden_layers={ANN_GRID['hidden_layers']}  "
          f"| activations={ANN_GRID['activations']}")
    print(f"        neurons={ANN_GRID['neurons']}  "
          f"| learning_rates={ANN_GRID['learning_rates']}")
    print(f"        dropout_rates={ANN_GRID['dropout_rates']}")

    best_score  = -np.inf
    best_params = {}
    results     = []

    for combo_i, (hl, act, neu, lr, dr) in enumerate(combos):
        fold_val_acc = []
        for tr_i, te_i in inner_skf.split(X_scaled, y):
            set_global_seed(RANDOM_STATE + combo_i)
            ann = ANNWrapper(
                input_dim    = input_dim,
                hidden_layers= hl,
                activation   = act,
                neurons      = neu,
                learning_rate= lr,
                dropout_rate = dr,
            )
            ann.fit(X_scaled[tr_i], y[tr_i])
            fold_val_acc.append(ann.score(X_scaled[te_i], y[te_i]))
            keras.backend.clear_session()
            gc.collect()

        mean_val_acc = np.mean(fold_val_acc)
        results.append({
            "hidden_layers": hl,
            "activation":    act,
            "neurons":       neu,
            "learning_rate": lr,
            "dropout_rate":  dr,
            "mean_val_acc":  mean_val_acc,
        })

        if (combo_i + 1) % 50 == 0 or (combo_i + 1) == n_combos:
            print(f"    combo {combo_i + 1}/{n_combos}  "
                  f"hl={hl} act={act} neu={neu} lr={lr} dr={dr}  "
                  f"val_acc={mean_val_acc:.4f}")

        if mean_val_acc > best_score:
            best_score = mean_val_acc
            best_params = {
                "hidden_layers": hl,
                "activation":    act,
                "neurons":       neu,
                "learning_rate": lr,
                "dropout_rate":  dr,
            }

    print(f"\n  [ANN] Best val_accuracy: {best_score:.4f}")
    print(f"  [ANN] Best params: {best_params}")

    # Save ANN grid results CSV
    ann_csv = pd.DataFrame(results).sort_values("mean_val_acc", ascending=False)
    ann_csv_path = os.path.join(OUTPUT_DIR, "tables", "ann_grid_search_results.csv")
    ann_csv.to_csv(ann_csv_path, index=False)
    print(f"  Saved → {ann_csv_path}")

    return best_params, best_score


# ============================================================================
# HYPERPARAMETER SEARCH — all 5 models
# ============================================================================
def run_hyperparameter_search(X_scaled: np.ndarray, y: np.ndarray) -> dict:
    """
    Run hyperparameter search for all 5 models:

      Logistic Regression — RandomizedSearchCV, plain accuracy, n_iter=500
      Decision Tree       — RandomizedSearchCV, plain accuracy, n_iter=2500
                            no gap penalty applied during search
      XGBoost             — RandomizedSearchCV, hard penalty (tol=10%)
                            fallback: 12% if all candidates fail
      Random Forest       — RandomizedSearchCV, hard penalty (tol=10%)
                            fallback: 12% if all candidates fail
      ANN                 — manual grid loop, best mean val_accuracy (3-fold)

    Model order follows ALL_MODELS:
      Logistic Regression → Decision Tree → XGBoost → Random Forest → ANN

    Returns dict of best_params per model.
    """
    global _ACTIVE_GAP_TOLERANCE

    print("\n" + "=" * 72)
    print("HYPERPARAMETER SEARCH — 5 Models")
    print(f"  Logistic Regression : plain accuracy          n_iter={TUNE_N_ITER_LR}")
    print(f"  Decision Tree       : plain accuracy          n_iter={TUNE_N_ITER_DT}")
    print(f"  XGBoost             : HARD penalty tol=10%    n_iter={TUNE_N_ITER_XGB}")
    print(f"  Random Forest       : HARD penalty tol=10%    n_iter={TUNE_N_ITER_RF}")
    print( "  ANN                 : manual grid  val_acc    "
          f"combos={len(list(itertools.product(*ANN_GRID.values())))}")
    print(f"  Gap scorer CV folds = {SCORER_CV_FOLDS}  |  "
          f"Search CV folds = {TUNE_CV_FOLDS}")
    print("=" * 72)

    X_vals = X_scaled

    # Configs for the four RandomizedSearchCV models in ALL_MODELS order
    search_configs = {
        "Logistic Regression": (build_lr(),  LR_PARAM_DIST,  TUNE_N_ITER_LR,  LR_SCORING,    "none"),
        "Decision Tree":       (build_dt(),  DT_PARAM_DIST,  TUNE_N_ITER_DT,  DT_SCORING,    "none"),
        "XGBoost":             (build_xgb(), XGB_PARAM_DIST, TUNE_N_ITER_XGB, RFXGB_SCORING, "hard"),
        "Random Forest":       (build_rf(),  RF_PARAM_DIST,  TUNE_N_ITER_RF,  RFXGB_SCORING, "hard"),
    }

    best_params  = {}
    summary_rows = []

    # Reset module-level state
    _ACTIVE_GAP_TOLERANCE = GAP_TOLERANCE

    # ── Loop over the 4 sklearn models in model-order ─────────────────────────
    for mname in ["Logistic Regression", "Decision Tree", "XGBoost", "Random Forest"]:
        estimator, param_dist, n_iter, scoring_fn, penalty_type = \
            search_configs[mname]
        fallback_used    = False
        active_tolerance = GAP_TOLERANCE

        print(f"\n  Tuning {mname}  (penalty={penalty_type}, n_iter={n_iter}) ...")

        if penalty_type == "none":
            # ── LR and DT: plain accuracy (no gap penalty) ────────────────────
            _ACTIVE_GAP_TOLERANCE = GAP_TOLERANCE

            rscv, best_score, _ = _run_single_search(
                clone(estimator), param_dist, n_iter, X_vals, y,
                scoring_fn=scoring_fn,
            )
            mean_train_acc, mean_test_acc, actual_gap = _diagnostic_cv(
                rscv.best_estimator_, X_vals, y
            )

        else:  # hard penalty — XGBoost and Random Forest
            # ── Hard penalty: RF / XGBoost ────────────────────────────────────
            _ACTIVE_GAP_TOLERANCE = GAP_TOLERANCE

            rscv, best_score, all_failed = _run_single_search(
                clone(estimator), param_dist, n_iter, X_vals, y,
                scoring_fn=scoring_fn,
            )
            active_tolerance = GAP_TOLERANCE

            if all_failed:
                print(f"\n    *** WARNING: all {n_iter} candidates for {mname} "
                      f"violated the {GAP_TOLERANCE:.0%} accuracy gap constraint.")
                print(f"    *** Re-running search with relaxed tolerance "
                      f"{GAP_FALLBACK_TOLERANCE:.0%} ...")
                _ACTIVE_GAP_TOLERANCE = GAP_FALLBACK_TOLERANCE
                rscv, best_score, still_failed = _run_single_search(
                    clone(estimator), param_dist, n_iter, X_vals, y,
                    scoring_fn=scoring_fn,
                )
                active_tolerance = GAP_FALLBACK_TOLERANCE
                fallback_used    = True
                if still_failed:
                    print(f"    *** WARNING: fallback at "
                          f"{GAP_FALLBACK_TOLERANCE:.0%} also failed for "
                          f"{mname}. Best available candidate selected.")

            mean_train_acc, mean_test_acc, actual_gap = _diagnostic_cv(
                rscv.best_estimator_, X_vals, y
            )

        # Reset for next model
        _ACTIVE_GAP_TOLERANCE = GAP_TOLERANCE

        bp         = rscv.best_params_
        best_params[mname] = bp
        gap_status = ("OK" if actual_gap <= active_tolerance
                       else "EXCEEDS TOLERANCE")
        penalty_std   = rscv.cv_results_["std_test_score"][rscv.best_index_]
        penalty_label = penalty_type  # "none" or "hard"

        print(f"    Penalty score (best candidate) : {best_score:.4f} "
              f"± {penalty_std:.4f}  [{penalty_label}]")
        print(f"    Diagnostic accuracy — "
              f"train: {mean_train_acc:.4f}  |  test: {mean_test_acc:.4f}  |  "
              f"gap: {actual_gap:.4f}  ({gap_status})"
              + ("  [FALLBACK]" if fallback_used else ""))
        print(f"    Best params: {bp}")

        row = {
            "Model":                       mname,
            "Penalty_type":                penalty_label,
            "n_iter":                      n_iter,
            "inner_cv_folds":              TUNE_CV_FOLDS,
            "scorer_cv_folds":             SCORER_CV_FOLDS,
            "GAP_TOLERANCE_used":          active_tolerance,
            "Soft_lambda_used":            "N/A",
            "fallback_triggered":          fallback_used,
            "Best_penalty_score_mean":     best_score,
            "Best_penalty_score_std":      penalty_std,
            "Diag_CV_mean_train_accuracy": mean_train_acc,
            "Diag_CV_mean_test_accuracy":  mean_test_acc,
            "Diag_CV_train_test_gap":      actual_gap,
            "Gap_status":                  gap_status,
        }
        for k, v in bp.items():
            row[f"param_{k}"] = v
        summary_rows.append(row)

    # ── ANN: manual grid search ───────────────────────────────────────────────
    print(f"\n  Tuning ANN  (manual grid, val_accuracy, {ANN_INNER_FOLDS}-fold inner CV) ...")
    ann_best_params, ann_best_val_acc = run_ann_grid_search(X_scaled, y)
    best_params["ANN"] = ann_best_params

    summary_rows.append({
        "Model":                       "ANN",
        "Penalty_type":                "none (val_accuracy)",
        "n_iter":                      len(list(itertools.product(*ANN_GRID.values()))),
        "inner_cv_folds":              ANN_INNER_FOLDS,
        "scorer_cv_folds":             ANN_INNER_FOLDS,
        "GAP_TOLERANCE_used":          "N/A",
        "Soft_lambda_used":            "N/A",
        "fallback_triggered":          False,
        "Best_penalty_score_mean":     ann_best_val_acc,
        "Best_penalty_score_std":      "N/A",
        "Diag_CV_mean_train_accuracy": "N/A",
        "Diag_CV_mean_test_accuracy":  ann_best_val_acc,
        "Diag_CV_train_test_gap":      "N/A",
        "Gap_status":                  "N/A",
        **{f"param_{k}": v for k, v in ann_best_params.items()},
    })

    # Save tuning summary CSV
    out_path = os.path.join(OUTPUT_DIR, "tables", "hyperparameter_tuning.csv")
    pd.DataFrame(summary_rows).to_csv(out_path, index=False)
    print(f"\n  Tuning summary saved → {out_path}")
    print("=" * 72)

    return best_params


# ============================================================================
# CORE CV LOOP  (5-fold × 10-repeat, all 5 models)
# ============================================================================
def run_cv(X, y, tuned_params: dict = None):
    """
    Execute 5-fold × 10-repeat stratified CV for all 5 models.
    Scaling is fitted on each fold's training portion independently.
    ANN uses tuned_params["ANN"] if provided, else analysis-file defaults.

    Returns:
        train_scores, test_scores   — per-fold metric dicts
        agg_preds, agg_true         — aggregated per-fold test arrays
    """
    tuned_params = tuned_params or {}

    rskf        = RepeatedStratifiedKFold(
        n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE
    )
    total_folds = N_SPLITS * N_REPEATS
    metrics_keys = ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]

    train_scores = {m: {k: [] for k in metrics_keys} for m in ALL_MODELS}
    test_scores  = {m: {k: [] for k in metrics_keys} for m in ALL_MODELS}
    agg_preds    = {m: [] for m in ALL_MODELS}
    agg_true     = {m: [] for m in ALL_MODELS}

    # Build sklearn model factories using tuned params
    def _make_builder(name, base_builder):
        bp = tuned_params.get(name, None)
        return (lambda bp=bp: base_builder(best_params=bp)) if bp \
               else (lambda: base_builder())

    sklearn_builders = {
        "Logistic Regression": _make_builder("Logistic Regression", build_lr),
        "Decision Tree":       _make_builder("Decision Tree",        build_dt),
        "XGBoost":             _make_builder("XGBoost",              build_xgb),
        "Random Forest":       _make_builder("Random Forest",        build_rf),
    }

    # ANN params: prefer tuned, fall back to analysis-file defaults
    ann_tuned = tuned_params.get("ANN", {})
    ann_hl    = ann_tuned.get("hidden_layers",  3)
    ann_act   = ann_tuned.get("activation",     "elu")
    ann_neu   = ann_tuned.get("neurons",        128)
    ann_lr    = ann_tuned.get("learning_rate",  0.0081)
    ann_dr    = ann_tuned.get("dropout_rate",   0.25)

    X_np = X.values if isinstance(X, pd.DataFrame) else X

    for fold_i, (tr_idx, te_idx) in enumerate(rskf.split(X_np, y)):
        print(f"\n  === Fold {fold_i + 1}/{total_folds} ===")

        scaler   = StandardScaler()
        X_tr_sc  = scaler.fit_transform(X_np[tr_idx])
        X_te_sc  = scaler.transform(X_np[te_idx])
        X_tr_df  = pd.DataFrame(X_tr_sc, columns=FEATURES)
        X_te_df  = pd.DataFrame(X_te_sc, columns=FEATURES)
        y_tr, y_te = y[tr_idx], y[te_idx]

        # ── sklearn models (LR, DT, XGB, RF) ──────────────────────────────────
        for mname, builder in sklearn_builders.items():
            model = builder()
            model.fit(X_tr_df, y_tr)

            yp_tr = model.predict(X_tr_df)
            for k, v in compute_metrics(y_tr, yp_tr).items():
                train_scores[mname][k].append(v)

            yp_te = model.predict(X_te_df)
            for k, v in compute_metrics(y_te, yp_te).items():
                test_scores[mname][k].append(v)

            agg_preds[mname].append(yp_te)
            agg_true[mname].append(y_te)

        # ── ANN ────────────────────────────────────────────────────────────────
        print(f"    ANN fold {fold_i + 1}/{total_folds} ...")
        ann = ANNWrapper(
            input_dim    = X_tr_sc.shape[1],
            hidden_layers= ann_hl,
            activation   = ann_act,
            neurons      = ann_neu,
            learning_rate= ann_lr,
            dropout_rate = ann_dr,
        )
        ann.fit(X_tr_sc, y_tr)

        yp_tr_ann = ann.predict(X_tr_sc)
        for k, v in compute_metrics(y_tr, yp_tr_ann).items():
            train_scores["ANN"][k].append(v)

        yp_te_ann = ann.predict(X_te_sc)
        for k, v in compute_metrics(y_te, yp_te_ann).items():
            test_scores["ANN"][k].append(v)

        agg_preds["ANN"].append(yp_te_ann)
        agg_true["ANN"].append(y_te)

        keras.backend.clear_session()
        gc.collect()

    agg_preds_out = {m: np.concatenate(agg_preds[m]) for m in ALL_MODELS}
    agg_true_out  = {m: np.concatenate(agg_true[m])  for m in ALL_MODELS}

    return train_scores, test_scores, agg_preds_out, agg_true_out


# ============================================================================
# FIGURE 1 — Model comparison bar plots
# Train & test bars: mean ± 1 SD across 50 per-fold scores
# Metrics: balanced_accuracy, accuracy, precision, recall, f1
# ============================================================================
def plot_model_comparison(train_scores: dict, test_scores: dict):
    print("\n[Figure 1] Model comparison bar plots ...")
    metrics       = ["balanced_accuracy", "accuracy", "precision", "recall", "f1"]
    metric_labels = {
        "balanced_accuracy": "Balanced Accuracy",
        "accuracy":          "Accuracy",
        "precision":         "Precision",
        "recall":            "Recall",
        "f1":                "F1 Score",
    }

    # Individual per-metric figures
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(10, 6))
        x     = np.arange(len(ALL_MODELS))
        bar_w = 0.35

        tr_means = [np.mean(train_scores[m][metric]) for m in ALL_MODELS]
        tr_sds   = [np.std(train_scores[m][metric], ddof=1) for m in ALL_MODELS]
        te_means = [np.mean(test_scores[m][metric])  for m in ALL_MODELS]
        te_sds   = [np.std(test_scores[m][metric],  ddof=1) for m in ALL_MODELS]

        model_colors = [COLORS[m] for m in ALL_MODELS]

        ax.bar(x - bar_w / 2, tr_means, bar_w,
               color=model_colors, alpha=0.85,
               yerr=tr_sds, capsize=4,
               error_kw=dict(elinewidth=1.4, ecolor="black", capthick=1.4),
               label="Train", zorder=3)
        ax.bar(x + bar_w / 2, te_means, bar_w,
               color=TEST_COLOR, alpha=0.80,
               yerr=te_sds, capsize=4,
               error_kw=dict(elinewidth=1.4, ecolor="black", capthick=1.4),
               label="Test", zorder=3)

        for i, mname in enumerate(ALL_MODELS):
            gap = tr_means[i] - te_means[i]
            ax.text(x[i],
                    max(tr_means[i], te_means[i]) + max(tr_sds[i], te_sds[i]) + 0.02,
                    f"Δ={gap:.2f}", ha="center", va="bottom", fontsize=8,
                    color="red" if gap > GAP_TOLERANCE else "green")

        ax.set_xticks(x)
        ax.set_xticklabels(ALL_MODELS, rotation=20, ha="right", fontsize=10)
        ax.set_ylabel(metric_labels[metric], fontsize=11)
        ax.set_ylim(0, 1.22)
        ax.set_title(f"Model Performance: {metric_labels[metric]}", fontsize=12)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)

        train_patch = mpatches.Patch(color="gray",     alpha=0.85,
                                     label="Train (mean ± 1 SD)")
        test_patch  = mpatches.Patch(color=TEST_COLOR, alpha=0.80,
                                     label="Test  (mean ± 1 SD)")
        ax.legend(handles=[train_patch, test_patch], fontsize=10, loc="upper left")
        ax.text(0.99, 0.02,
                f"Δ = train − test  |  red = gap > {GAP_TOLERANCE:.0%}  "
                f"|  LR/DT: plain acc  |  RF/XGB: hard  |  ANN: val_acc grid",
                transform=ax.transAxes, fontsize=7, ha="right", va="bottom",
                color="gray")

        n_folds = N_SPLITS * N_REPEATS
        plt.suptitle(
            f"Model Performance: {metric_labels[metric]} — "
            f"{N_SPLITS}-Fold × {N_REPEATS}-Repeat CV (n={n_folds} folds)",
            fontsize=12, y=1.01,
        )
        plt.tight_layout()
        savefig(fig, f"model_comparison/Fig1_{metric}")
        plt.close(fig)

    # Combined 2×3 figure (5 metrics + 1 blank panel)
    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    axes      = axes.flatten()
    x         = np.arange(len(ALL_MODELS))
    bar_w     = 0.35

    for idx, metric in enumerate(metrics):
        ax       = axes[idx]
        tr_means = [np.mean(train_scores[m][metric]) for m in ALL_MODELS]
        tr_sds   = [np.std(train_scores[m][metric], ddof=1) for m in ALL_MODELS]
        te_means = [np.mean(test_scores[m][metric])  for m in ALL_MODELS]
        te_sds   = [np.std(test_scores[m][metric],  ddof=1) for m in ALL_MODELS]

        model_colors = [COLORS[m] for m in ALL_MODELS]

        ax.bar(x - bar_w / 2, tr_means, bar_w,
               color=model_colors, alpha=0.85,
               yerr=tr_sds, capsize=4,
               error_kw=dict(elinewidth=1.4, ecolor="black", capthick=1.4),
               zorder=3)
        ax.bar(x + bar_w / 2, te_means, bar_w,
               color=TEST_COLOR, alpha=0.80,
               yerr=te_sds, capsize=4,
               error_kw=dict(elinewidth=1.4, ecolor="black", capthick=1.4),
               zorder=3)

        for i, mname in enumerate(ALL_MODELS):
            gap = tr_means[i] - te_means[i]
            ax.text(x[i],
                    max(tr_means[i], te_means[i]) + max(tr_sds[i], te_sds[i]) + 0.02,
                    f"Δ={gap:.2f}", ha="center", va="bottom", fontsize=7,
                    color="red" if gap > GAP_TOLERANCE else "green")

        ax.set_xticks(x)
        ax.set_xticklabels(ALL_MODELS, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel(metric_labels[metric], fontsize=10)
        ax.set_ylim(0, 1.22)
        ax.set_title(metric_labels[metric], fontsize=11)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)

    axes[5].set_visible(False)   # blank 6th panel

    train_patch = mpatches.Patch(color="gray",     alpha=0.85,
                                 label="Train (mean ± 1 SD)")
    test_patch  = mpatches.Patch(color=TEST_COLOR, alpha=0.80,
                                 label="Test  (mean ± 1 SD)")
    fig.legend(handles=[train_patch, test_patch],
               fontsize=10, loc="lower right", bbox_to_anchor=(0.98, 0.05))

    n_folds = N_SPLITS * N_REPEATS
    plt.suptitle(
        f"Model Performance Comparison — {N_SPLITS}-Fold × {N_REPEATS}-Repeat CV "
        f"(n={n_folds} folds)  |  Train & Test mean ± 1 SD\n"
        f"Δ annotation: red = gap > {GAP_TOLERANCE:.0%}  |  green = gap ≤ {GAP_TOLERANCE:.0%}  "
        f"|  LR/DT: plain acc  |  RF/XGB: hard  |  ANN: val_acc grid",
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    savefig(fig, "model_comparison/Fig1_model_comparison_combined")
    plt.close(fig)


# ============================================================================
# FIGURE 2 — Confusion matrices
# ============================================================================
def _draw_cm(ax, cm, labels, title):
    cmap = plt.cm.Blues
    ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.set_title(f"Confusion Matrix\n{title}", fontsize=11, pad=10)
    tick_marks = np.arange(len(labels))
    ax.set_xticks(tick_marks);     ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticks(tick_marks);     ax.set_yticklabels(labels, fontsize=10,
                                                       rotation=90, va="center")
    ax.set_xlabel("Predicted Class", fontsize=10)
    ax.set_ylabel("True Class",      fontsize=10)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > thresh else "black"
            ax.text(j, i, str(cm[i, j]),
                    ha="center", va="center",
                    fontsize=14, fontweight="bold", color=color)


def plot_confusion_matrices(agg_preds: dict, agg_true: dict):
    print("\n[Figure 2] Confusion matrices ...")
    class_labels = ["Low Risk", "Risky"]

    # Individual per-model figures
    for mname in ALL_MODELS:
        cm  = confusion_matrix(agg_true[mname], agg_preds[mname])
        fig, ax = plt.subplots(figsize=(5, 4))
        _draw_cm(ax, cm, class_labels, mname)
        plt.tight_layout()
        safe = mname.replace(" ", "_")
        savefig(fig, f"confusion_matrix/Fig2_CM_{safe}")
        plt.draw(); plt.close(fig); plt.clf(); plt.cla()
        gc.collect()

    # Combined 2×3 figure (5 models + 1 blank panel)
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    axes = axes.flatten()
    for i, mname in enumerate(ALL_MODELS):
        cm = confusion_matrix(agg_true[mname], agg_preds[mname])
        _draw_cm(axes[i], cm, class_labels, mname)
    axes[5].axis("off")

    plt.suptitle(
        f"Confusion Matrices — All Models  "
        f"(Aggregated per-fold test: {N_SPLITS}-Fold × {N_REPEATS}-Repeat CV)",
        fontsize=14, y=1.01,
    )
    plt.tight_layout()
    savefig(fig, "confusion_matrix/Fig2_CM_combined")
    plt.draw(); plt.close(fig); plt.clf(); plt.cla()
    gc.collect()


# ============================================================================
# METRICS SUMMARY TABLE
# ============================================================================
def save_metrics_table(train_scores: dict, test_scores: dict):
    rows = []
    for mname in ALL_MODELS:
        row = {"Model": mname}
        for met in ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]:
            tr = np.array(train_scores[mname][met])
            te = np.array(test_scores[mname][met])
            row[f"Train_CV_{met}_mean"] = np.mean(tr)
            row[f"Train_CV_{met}_std"]  = np.std(tr, ddof=1)
            row[f"Test_CV_{met}_mean"]  = np.mean(te)
            row[f"Test_CV_{met}_std"]   = np.std(te, ddof=1)
            row[f"Gap_{met}"]           = np.mean(tr) - np.mean(te)
        rows.append(row)
    path = os.path.join(OUTPUT_DIR, "tables", "metrics_summary.csv")
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"  Saved → {path}")


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("\nTensorFlow version:", tf.__version__)
    print("Matplotlib backend:", matplotlib.get_backend())
    set_global_seed(RANDOM_STATE)
    setup_output_dirs()

    print("\nCurrent working directory:", os.getcwd())
    print("Absolute output directory: ", os.path.abspath(OUTPUT_DIR))

    # Load data
    X, y = load_data()

    # Scale full dataset once for hyperparameter search
    print("\n[Tuning] Scaling full dataset for hyperparameter search ...")
    X_np_full     = X.values if isinstance(X, pd.DataFrame) else X
    scaler_tune   = StandardScaler()
    X_scaled_tune = scaler_tune.fit_transform(X_np_full)

    # Hyperparameter search — run ONCE on full scaled dataset
    tuned_params = run_hyperparameter_search(X_scaled_tune, y)

    # Full CV loop with tuned params
    print(f"\n[CV] Running {N_SPLITS}-fold × {N_REPEATS}-repeat stratified CV "
          f"({N_SPLITS * N_REPEATS} total folds) ...")
    print("     All 5 models use tuned hyperparameters from the search above.")
    print("     LR: plain accuracy  |  DT: plain accuracy  |  "
          "RF/XGB: hard penalty  |  ANN: manual grid val_acc")
    train_scores, test_scores, agg_preds, agg_true = run_cv(
        X, y, tuned_params=tuned_params
    )

    # Metrics table
    print("\n[Tables] Saving metrics summary ...")
    save_metrics_table(train_scores, test_scores)

    # Figures
    plot_model_comparison(train_scores, test_scores)
    plot_confusion_matrices(agg_preds, agg_true)

    print("\n[OK]  All figures saved to:", os.path.join(OUTPUT_DIR, "figures"))
    print("   Metrics table      saved to:", os.path.join(OUTPUT_DIR, "tables",
                                                           "metrics_summary.csv"))
    print("   Tuning summary     saved to:", os.path.join(OUTPUT_DIR, "tables",
                                                           "hyperparameter_tuning.csv"))
    print("   ANN grid results   saved to:", os.path.join(OUTPUT_DIR, "tables",
                                                           "ann_grid_search_results.csv"))


if __name__ == "__main__":
    main()
